# Module 7.1: PEFT and LoRA (Low-Rank Adaptation)

Welcome to Phase 3: Applied LLM Engineering! We know how to build and train Transformers, but there is an elephant in the room: **Hardware**.

If you want to Supervised Fine-Tune (SFT) a 70 Billion parameter model on your own dataset, doing standard backpropagation requires hundreds of gigabytes of VRAM. Nobody has that at home.

In this notebook, we look at answering three questions: **What** is PEFT, **Why** is it needed, and **How** does LoRA perform mathematical black magic to solve the memory crisis?

## 1. WHAT is PEFT?

**Parameter-Efficient Fine-Tuning (PEFT)** is the blanket term for methods that train a massive model without touching the original weights.

### \ud83d\udcdd The Dictionary Analogy
Imagine you own a heavy, 5,000-page Encyclopedia (our 70B parameter Base Model). 
You want to learn some new 2024 internet slang (Fine-Tuning).

**Standard Fine-Tuning**: You rip out every page of the encyclopedia, rewrite the definitions, and re-bind the entire heavy book. This takes massive effort (compute/memory).

**PEFT**: You take a tiny, lightweight stack of sticky notes. You write the new slang on the sticky notes and paste them onto the back cover of the Encyclopedia. When you need a word, you check the sticky notes first, then check the book. The heavy book stays completely untouched (frozen)!

## 2. WHY do we need LoRA?

When you use standard fine-tuning, you load the Model Weights. But when PyTorch calculates gradients (the slopes) and optimizer states (AdamW momentum), it creates *new matrices* that are the exact same size as the model.

If the Model is $10$ GB, the gradients are $10$ GB, and the optimizer states are $20$ GB. Total memory = **$40$ GB**! Just to train!

**LoRA (Low-Rank Adaptation)** is the most popular form of PEFT. It slashes the training memory by up to 90% because it only creates gradients and optimizer states for the tiny sticky notes, not the massive encyclopedia!

## 3. HOW does LoRA work? (The Math)

LoRA works by hijacking the massive dense linear layers (`nn.Linear`) in the Attention blocks.

Instead of updating the master weight matrix $W$, it freezes $W$. It then creates two tiny matrices: **$A$** and **$B$**. 
By multiplying $B \times A$, we reconstruct a matrix the exact size of $W$, but with millions of fewer parameters to track!

### \ud83d\udeb0 The Pipe Analogy (Low-Rank Compression)
Imagine pouring a giant lake of water (Matrix $A$ inputs) through an ultra-thin garden hose (The bottleneck "Rank"), and then spraying it back out to fill another lake (Matrix $B$ outputs). The "Rank" $r$ determines how thin the hose is. A thinner hose means less memory used, but less information can pass through.

```mermaid
graph TD
    Input[Tokens X] --> W[Frozen Model Weights W\nDim: 4096 x 4096\nParams: 16 Million!]
    
    Input -.->|LoRA Injection| A[Matrix A \nDim: 4096 x 'r']
    A -.->|The 'r' bottleneck\n( e.g. r=8 )| B[Matrix B \nDim: 'r' x 4096]
    
    W --> Sum{+}
    B -.->|Params: 65 Thousand!| Sum
    
    Sum --> Out[Output]
```

In [ ]:
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=8, alpha=16):
        super().__init__()
        
        # 1. The Encyclopedia (The original massive frozen weights)
        self.W = nn.Linear(in_features, out_features, bias=False)
        self.W.weight.requires_grad = False # FREEZE! No memory needed for gradients here!
        
        # 2. The Sticky Notes (The LoRA A and B matrices)
        # Notice how input shrinks down to `rank`, then expands back to `out_features`
        self.A = nn.Linear(in_features, rank, bias=False)
        self.B = nn.Linear(rank, out_features, bias=False)
        
        # Alpha is just a scaling factor (how 'loud' the sticky notes are compared to the book)
        self.scaling = alpha / rank 
        
        # Zero-initialize B so that at the start of training, LoRA outputs exactly 0
        # This ensures the model acts EXACTLY like the frozen base model until training begins!
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        # Forward pass through the frozen base model
        frozen_out = self.W(x)
        
        # Forward pass through the tiny LoRA pipeline
        lora_out = self.B(self.A(x)) * self.scaling
        
        # Add the sticky notes to the encyclopedia!
        return frozen_out + lora_out


# Let's visualize the parameter savings!
d_model = 4096
lora_layer = LoRALinear(in_features=d_model, out_features=d_model, rank=8)

frozen_params = sum(p.numel() for p in lora_layer.W.parameters())
trainable_params = sum(p.numel() for p in lora_layer.A.parameters()) + sum(p.numel() for p in lora_layer.B.parameters())

print(f"Original Model Params (Frozen):   {frozen_params:,}")
print(f"LoRA Trainable Params  (Rank 8):  {trainable_params:,}")
print(f"\nWe reduced the training memory bottleneck by {((frozen_params - trainable_params) / frozen_params) * 100:.2f}%!!")

## Summary

By freezing the 16+ million parameter matrix ($W$) and only training the 65 thousand parameter matrices ($A$ and $B$), the optimizer doesn't need to save history for the massive bulk of the model! 

When you download a "LoRA Adapter" from the internet (which is usually a tiny $100$ MB file instead of a $20$ GB model), you are literally downloading these tiny $A$ and $B$ sticky-note matrices, and dynamically merging them onto your base model at runtime!